### IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore") # Suppress warnings for cleaner output

### LOAD DATA

In [ ]:
df = pd.read_csv("../data/household_power_consumption.txt", sep=";", na_values=["?"], low_memory=False)
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
df = df.drop(columns=["Date", "Time"])
df = df.set_index("datetime").sort_index()
df.head()

### PLOT THE MAIN SIGNAL: GLOBAL ACTIVE POWER

In [ ]:
# perform daily sampling
daily = df["Global_active_power"].resample("D").mean()

# plot the main signal
plt.figure(figsize=(12, 6))
plt.plot(daily, color ="steelblue", linewidth=0.8)
plt.title("Daily Average Global Active Power")
plt.xlabel("Date")
plt.ylabel("kW")
plt.tight_layout()
plt.show()

### ZOOM INTO 1 MONTH AND ANALYZE

In [ ]:
# zoom into Jan 2008
sample = df["Global_active_power"]["2008-01"].resample("h").mean()

# Plot the zoomed-in signal
plt.figure(figsize=(14,4))
plt.plot(sample, color="steelblue", linewidth=1)
plt.title("Hourly Average Global Active Power (January 2008)")
plt.xlabel("Date")
plt.ylabel("kW")
plt.tight_layout()
plt.show()

# Print some basic statistics
print(sample.describe())
print("\nZero values in the sample:", (sample == 0).sum())
print("Missing values in the sample:", sample.isna().sum())

### CHARACTERIZE THE ANOMALIES WE'LL BE TARGETING

In [ ]:
# Anomaly type 1: Missing data gaps
gaps = df["Global_active_power"].resample("h").mean()
missing_blocks = gaps[gaps.isna()]
print("Hours with missing data:", len(missing_blocks))

# Anomaly type 2: Spikes (values > 3 STD from rolling mean)
hourly = df["Global_active_power"].resample("h").mean()
rolling_mean = hourly.rolling(window=24, center=True).mean()
rolling_std = hourly.rolling(window=24, center=True).std()
spikes = hourly[(hourly - rolling_mean).abs() > 3 * rolling_std]
print("Number of spikes detected:", len(spikes))

# Anomaly type 3: Flatlines (zero variance over a window)
rolling_var = hourly.rolling(window=24, center=True).var()
flatlines = hourly[rolling_var < 0.001]  # Threshold for flatline detection
print("Number of flatlines detected:", len(flatlines))
